# Aperiodic sample: universal retrieval across the hysteresis loop

Run `00_preprocess_2871_2872.ipynb` once. This notebook reads its repository cache and jointly retrieves the saturated reference and selected hysteresis states at one energy and helicity. One known saturation anchors a relative magnetic decomposition; the absolute charge baseline and magnetic amplitude remain ambiguous without additional constraints.

Every selected scan-2924 point is a separate magnetic state, including repeated field values on different hysteresis branches. The default selects scan-2924 points 1–10 (10 magnetic field states, each averaged from 100 CMOS frames); point 0 repeats the approximately +1 A saturated endpoint from scan 2923. Set `FIRST_N_STATES = 10` to retrieve only the first 10 field points, or `POINT_INDICES = "preview"` for up to 10 points spread across the loop.

Set `MODES = [1, 2]` for two incoherent modes: mode 2 has a 2× enlarged support around each aperture, while the charge/magnetic/thickness projection applies only to mode 1.


Set the scan numbers in the first settings cell to match notebook 00. The support is loaded from its preprocessing cache.

Detector exclusions now come from the scan-pair painted PNG only, plus automatic saturation exclusions when stored separately in a new cache. JSON polygon coordinates, legacy combined masks, implicit dilation and hard-coded excluded columns are not applied. Re-run 00 to preserve automatic saturation masks when using an older cache.


### Optional live reconstruction
Set `LIVE_RECONSTRUCTION=True` to update magnetization and each mode’s focused exit-wave amplitude/phase after physical projections. `LIVE_OBSERVATIONS=None` selects saturation and the comparison observation; provide indices to select others. `LIVE_EVERY_PROJECTION` and `LIVE_MIN_SECONDS` limit redraw frequency. The first image appears after the first physical projection, not during independent warmup. A final image is always shown. Magnetization is the latest fitted model, while exit waves are current fields; amplitude color limits adjust on each update. This is display-only and adds plotting/FFT overhead.


## Workflow controls: what runs, and what to change

1. **Prepare data:** center the detector, mask invalid pixels, then crop and bin. The retrieval grid and aperture ROI follow those choices.
2. **Initialize:** FFT of the support and global scaling; `STARTIMAGE_RADIAL_NORMALIZATION` optionally replaces the intensity envelope with the valid-pixel radial average. It is an initial guess, not an iterative constraint.
3. **Warm up:** reconstruct each observation coherently. With partial coherence enabled, capture its masked diffraction intensities before the partial-coherence pass. Reference seeding, where configured, transfers phase information to other observations.
4. **Joint rounds:** detector/support updates alternate with the physical model. Only mode 1 has magnetic components. Mode 2 is made common across states/polarizations within an energy/illumination group at projections and on output.
5. **Optional coherent refresh:** enable partial coherence and set `REFRESH_MASK_AFTER_OUTER_LOOPS = [3, 7]` (requires at least eight outer rounds). After those completed loops, run `COHERENT_REFRESH_MODE` for `COHERENT_REFRESH_ITERATIONS`, with masked pixels free and blur disabled. Replace the masked fills with these results, retain gamma, and resume partial coherence. `[]` preserves the original workflow. Frozen reference observations stay frozen.
6. **Inspect/refine:** compare warmup, joint result, and optional quasi-Poisson refinement. Refinement changes the primary field, so baseline physical maps do not describe the refined result.

**To change runtime:** edit `outer_iterations`, `inner_Nit`, warmup iteration counts, observation workers and FFT workers in the recipe/configuration cells. More workers require more memory. **To change coupling:** edit `projection_relaxation`, `projection_every`, and physical/spectral constraints. `final_projection_relaxation=0` avoids an extra final physical update. **To change coherence:** use `USE_PARTIAL_COHERENCE`, kernel scope, shared-gamma strategy and RL controls. A shared gamma with `average` fits each observation independently before averaging; `pooled` fits all observations jointly.

The text tree and figure immediately before retrieval show the resolved workflow. Changing a configuration cell requires rerunning the recipe cell. Restart the kernel after library edits. See [the detailed library guide](../docs/universal_phase_retrieval.md) for API, option reference, transition semantics and examples.


### Independent or reference-seeded warmup

The default here is **independent warmup for every observation**: `WARMUP_START_FROM_REFERENCE=False`. This includes both opposite polarizations; neither is seeded from the other. Every observation uses its own support-based, normalized start (and optional radial normalization), then full-coherence **HAPRE × 750 + ER × 50**. Independent means no reconstructed reference is copied; identical supports may still have identical initial phases. Physical projections subsequently couple the primary fields, with phase-reference handling where enabled; arbitrary spatial phase ambiguities are not guaranteed to resolve.

Set `WARMUP_START_FROM_REFERENCE=True` to retrieve the selected reference first and seed the remaining observations from its scaled field. In ajajas, keep `WARMUP_POLICY="same_for_all"` for the same 750+50 schedule, or select `"reference_and_others"` to use the separate `OTHER_WARMUP_*` schedule. Shared gamma calibration is a separate setting from field seeding.

### Refresh missing diffraction after chosen outer loops

Set `REFRESH_MASK_AFTER_OUTER_LOOPS=[3, 7]` to update the missing diffraction after completing loops 3 and 7, before the next partial-coherence updates. The short coherent pass frees `mask_pixel==1` pixels, then their resulting intensities become the new fixed fills. Gamma is retained. Set its duration with `COHERENT_REFRESH_ITERATIONS`. Leave the list empty to disable it. Entries must be completed loop numbers **smaller than `outer_iterations`**; full coherence alone already leaves masked pixels free, so refresh requires partial coherence enabled. Frozen reference fields remain unchanged.


In [ ]:
from pathlib import Path
import sys, json
import h5py
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "00_preprocess_2871_2872.ipynb").is_file():
    HERE = HERE / "aperiodic_ajajas"
if not (HERE / "00_preprocess_2871_2872.ipynb").is_file():
    raise FileNotFoundError("Run from the repository root or aperiodic_ajajas")
REPO = HERE.parent
print(REPO)
sys.path.insert(0, str(REPO))
# Match the scan numbers used in notebook 00.

SATURATED_SCAN = 2923
LOOP_SCAN = 2924

SATURATED_SCAN = 2871
LOOP_SCAN = 2872

OUTPUT_DIR = HERE / "processed" / f"scans_{SATURATED_SCAN}_{LOOP_SCAN}"
from aperiodic_ajajas.result_io import prepare_painted_detector_mask, painted_cache_mask
MASK_PIXEL_PNG = HERE / f"mask_pixel_{SATURATED_SCAN}.png"
from library import fthcore as fth
from library.phase_retrieval_geometry import recenter_source_support
from aperiodic_ajajas.reconstruction_inputs import (
    adapt_retrieval_array_shapes, center_crop_or_pad,
    crop_array_edges,
    detector_file_point_indices, fit_reference_intensity,
)
CACHE = HERE / "processed" / f"preprocessed_{LOOP_SCAN}_{SATURATED_SCAN}.h5"
if not CACHE.is_file():
    raise FileNotFoundError(f"Run 00_preprocess_2871_2872.ipynb first: {CACHE}")

SATURATED_SIGN = +1  # Set to -1 if 2923 was negatively saturated.
SUPPORT_SHIFT = (0, 0)  # Cached support already aligns with scan 2923; (row, column) pixels.
POINT_INDICES = None  # None uses the regular stride selection; an explicit list remains exact.
STATE_STRIDE = 1  # 1 uses every state, 2 every second state, 3 every third state, and so on.
FIRST_N_STATES = None  # Optional limit applied after STATE_STRIDE.
MASK_REFERENCE_POINT_NUMBER = 35  # Current STATE_INDEX=7 comparison in notebook 01.
ALWAYS_INCLUDE_COMPARISON_STATE = True  # Add this point if stride/limit omitted it.
PREVIEW_STATES = 15  # Number of hysteresis points if using "preview" (capped by scan length).
RETRIEVAL_CROP = 700  # Must match notebook 01 for an identical warmup grid.
RETRIEVAL_BINNING = 1  # Crop first, then bin; notebook 01 currently uses no binning.
NORMALIZE_LOOP_INTENSITY = False  # Match 01/Copy2 during warmup; fitted factors stay diagnostic.
HOLOGRAM_OFFSET = 0.0  # Copy2 input offset; fitted intercept is diagnostic only.
FILTER_EDGE_PIXELS = 0  # Set these object-space edge pixels to zero before recalculating holograms.
FILTER_TEMP_CROP_PIXELS = 0  # Optionally remove this many more temp pixels per edge; 0 disables cropping.
SUPPORT_FROM_01 = OUTPUT_DIR / "two_state_point_0050.h5"  # Result saved by 01; change point if needed.
# Shared focus settings for rec.focusCDI/display_object and every physical projection.
# DISPLAY_PROP_UM uses the same micrometre convention as the focusCDI slider.
# Set DISPLAY_PROP_UM = 0 to skip focus propagation and phase rotation in the projector.
DISPLAY_PROP_UM = 3.22+0*2.05
DISPLAY_PHASE_RAD = -0.76
DISPLAY_SETUP_BASE = {"ccd_dist": 0.125, "px_size": 11e-6}
DIAGNOSTIC_ROI_OVERRIDE = None  # Optional np.s_[rows, columns] on the retrieval grid.
DIAGNOSTIC_ROI_PADDING = 0.15  # Automatic ROI: largest aperture plus fractional padding.
PHYSICAL_PROJECTION_OBJECT_ROI = True  # Fit only material-aperture pixels; preserve the exterior.
PHYSICAL_PHASE_REFERENCE = True  # Use nearest phase branch relative to saturation, per energy/beam.
RUN_FOCUS_WIDGETS = True
LIVE_RECONSTRUCTION = False  # True: update images during physical projections.
LIVE_OBSERVATIONS = None  # None: saturation + comparison; or indices such as [0, 1].
LIVE_EVERY_PROJECTION = 1
LIVE_MIN_SECONDS = 2.0  # Throttle display work; final output always shown.
MODES = [1,2]  # Match notebook 01; use [1] in both notebooks for a single-mode comparison.
CONSTRAIN_OTHER_MODES_COMMON = True  # Mode 2 has no state-dependent component.
OTHER_MODES_COMMON_RELAXATION = 1.0  # Make mode 2 identical across states at each projection.
USE_PARTIAL_COHERENCE = False # Fit detector blur during warmup and joint retrieval.
# Warmup control: "same_for_all" applies REFERENCE_WARMUP_* to every hologram.
# "reference_and_others" applies REFERENCE_WARMUP_* to saturated and OTHER_WARMUP_* to every loop state.
WARMUP_POLICY = "reference_and_others"
WARMUP_START_FROM_REFERENCE = True  # False gives every hologram its own support-based start image.
REFERENCE_WARMUP_MODE = ["HAPRE", "ER"]
REFERENCE_WARMUP_NIT = [750, 80]
OTHER_WARMUP_MODE = ["HAPRE", "ER"]
OTHER_WARMUP_NIT = [750,80]
USE_COMMON_COHERENCE_KERNEL = True
SHARED_COHERENCE_UPDATE = "pooled"  # "average", "pooled", or legacy "sequential".
WARMUP_CALIBRATE_SHARED_GAMMA = True  # Fit saturation first; seed every loop PC warmup with that gamma.
OBSERVATION_WORKERS = 4  # Large-grid multimode PC uses several GiB per worker; benchmark before increasing.
FFT_WORKERS = 1  # Per-state FFT threads; avoid workers × threads exceeding CPU cores.
COHERENCE_ROUND_ITERATIONS = 50  # Pooled RL steps; also gamma-only fits of frozen fields.
COHERENCE_KERNEL_SCOPE = "shared" if USE_COMMON_COHERENCE_KERNEL else "per_observation"
PRESERVE_WARMUP_MASKED_INTENSITY = True  # All states finish coherent warmup before any partial-coherence stage.
FREEZE_SATURATED_FIELD = True  # Keep the well-reconstructed 2923 field fixed during joint physical updates.
FIX_COHERENCE_AFTER_SATURATED = False  # True selects sequential reference-only gamma fitting.
MODE_INITIALIZATION = "support_fft"  # Match notebook 01 instead of random multimode Fourier phases.
MODE_INITIALIZATION_SEED = 0
STARTIMAGE_RADIAL_NORMALIZATION = False  # Optional radial envelope after FFT + global scaling.
MODE_SUPPORT_CENTER = "image"  # Scale the entire support, including aperture separations.
THICKNESS_APERTURE_MODE = "fixed"  # "none", "support" (fit inside largest aperture), or "fixed" (binary thickness).
MATERIAL_MASK_FILE = None  # Optional PNG or .npy: 1 material, 0 known vacuum holes.
RELATIVE_THICKNESS_FILE = None  # Optional nonnegative .npy on the cached object grid.
FIT_MATERIAL_THICKNESS = False  # Fit a shared nonnegative relative thickness field.
ZERO_THICKNESS_OUTSIDE_SUPPORT = False  # Force vacuum outside the phase-retrieval support.
OUTPUT = OUTPUT_DIR / "universal.h5"

PNG_OUTPUT = OUTPUT.with_suffix("") / "reconstructions_png"

# Optional independent primary-field polishing; baseline fields stay unchanged.
RUN_POISSON_REFINEMENT = True
POISSON_STEPS = 50
POISSON_LEARNING_RATE = 1.0  # Backtracking reduces this if needed.

# Optional PC -> coherent -> PC cycles after completed outer loops.
# Example: [3, 7] refreshes masked diffraction before rounds 4 and 8.
REFRESH_MASK_AFTER_OUTER_LOOPS = []  # e.g. [3, 7]: refresh after completed loops 3 and 7.
COHERENT_REFRESH_ITERATIONS = 50
COHERENT_REFRESH_MODE = "ER"

# Final physical-unit conversion only: does not change reconstruction.
PHASE_PROPAGATION_SIGN = -1  # exp(-i*k*(n-1)*d); +1 supports the opposite convention.
COBALT_THICKNESS_NM = None  # Enter actual total Co thickness in nm; None leaves log-response units.
NORMALIZE_MAGNETIZATION_TO_UNIT_RANGE = False  # True: global max |m| in material becomes 1.

LOOP_EXCLUDE_POINT_NUMBERS = [0]
RETRIEVAL_POINT_NUMBERS = None
MASK_COLUMNS = []

def select_numbers(available, requested=None, step=1, exclude=()):
    """Return positions in available, selecting acquisition numbers, not positions."""
    available = [int(v) for v in available]
    if len(set(available)) != len(available):
        raise ValueError('Duplicate acquisition point numbers')
    if requested is None:
        if type(step) is not int or step < 1:
            raise ValueError('Selection step must be a positive integer')
        result = [i for i, value in enumerate(available) if value not in exclude][::step]
    else:
        if not requested or any(type(v) is not int or v < 0 for v in requested):
            raise ValueError('Selection must be a nonempty list of nonnegative integers')
        if len(set(requested)) != len(requested):
            raise ValueError('Duplicate requested indices')
        missing = set(requested) - set(available)
        if missing:
            raise ValueError(f'Requested indices are unavailable: {sorted(missing)}')
        result = [available.index(v) for v in requested]
    if not result:
        raise ValueError('Selection is empty')
    return result


## Read all selected states from the prepared cache


In [ ]:
with h5py.File(CACHE) as h:
    cache_config = json.loads(h.attrs["config_json"])
    if (cache_config.get("saturated_scan") != SATURATED_SCAN
            or cache_config.get("loop_scan") != LOOP_SCAN):
        raise ValueError("Cache scan numbers do not match notebook settings")
    if (cache_config.get("crop_raw_pixels") != 0
            or cache_config.get("binning") != 1
):
        raise ValueError("Cached preprocessing uses the old geometry or pixel mask. Re-run 00_preprocess_2871_2872.ipynb.")
    ENERGY_EV = float(h.attrs["photon_energy_eV"])
    DISPLAY_SETUP = {**DISPLAY_SETUP_BASE, "energy": ENERGY_EV}
    n_states = h["loop_holograms"].shape[0]
    loop_point_indices = detector_file_point_indices(h["loop_file_names"].asstr()[:])
    if len(loop_point_indices) != n_states:
        raise ValueError("Loop file list and hologram stack have different lengths")
    # Point 0 is the +1 A saturated endpoint already represented by scan 2923.
    # Each cached point averages all 100 CMOS frames from one Marana file.
    default_indices = select_numbers(loop_point_indices, exclude=LOOP_EXCLUDE_POINT_NUMBERS)
    if RETRIEVAL_POINT_NUMBERS is not None:
        POINT_INDICES = select_numbers(loop_point_indices, RETRIEVAL_POINT_NUMBERS)
    if isinstance(STATE_STRIDE, (bool, np.bool_)) or not isinstance(STATE_STRIDE, (int, np.integer)) or STATE_STRIDE < 1:
        raise ValueError("STATE_STRIDE must be a positive integer")
    if FIRST_N_STATES is not None:
        if not isinstance(FIRST_N_STATES, (int, np.integer)) or FIRST_N_STATES < 1:
            raise ValueError("FIRST_N_STATES must be a positive integer or None")
        if POINT_INDICES is not None:
            raise ValueError("Set either FIRST_N_STATES or POINT_INDICES, not both")
    if POINT_INDICES == "preview":
        indices = [default_indices[i] for i in np.unique(np.linspace(
            0, len(default_indices) - 1, min(PREVIEW_STATES, len(default_indices)), dtype=int,
        ))]
        selection_description = f"preview ({len(indices)} states spread over the loop)"
    elif POINT_INDICES is None:
        indices = default_indices[::int(STATE_STRIDE)]
        if FIRST_N_STATES is not None:
            indices = indices[:FIRST_N_STATES]
        selection_description = f"every {int(STATE_STRIDE)} state(s)"
        if FIRST_N_STATES is not None:
            selection_description += f", limited to the first {FIRST_N_STATES}"
    else:
        indices = list(POINT_INDICES)
        selection_description = "explicit POINT_INDICES"
    reference_matches = np.flatnonzero(loop_point_indices == MASK_REFERENCE_POINT_NUMBER)
    if len(reference_matches) != 1:
        raise ValueError("MASK_REFERENCE_POINT_NUMBER is absent or duplicated in the cache")
    comparison_cache_index = int(reference_matches[0])
    if comparison_cache_index not in indices:
        if ALWAYS_INCLUDE_COMPARISON_STATE:
            indices.append(comparison_cache_index)
            indices.sort()
            print(
                f"Added scan-{LOOP_SCAN} point {MASK_REFERENCE_POINT_NUMBER} for the mask comparison; "
                "set ALWAYS_INCLUDE_COMPARISON_STATE=False to require an exact selection."
            )
        else:
            raise ValueError(
                "The selected states omit MASK_REFERENCE_POINT_NUMBER. Choose a selected "
                "comparison point or set ALWAYS_INCLUDE_COMPARISON_STATE=True."
            )
    comparison_observation = 1 + indices.index(comparison_cache_index)
    warmup_reference_observation = 0  # Saturated scan 2923 is reconstructed first.
    if len(indices) < 2 or len(set(indices)) != len(indices):
        raise ValueError("Select at least two distinct loop points")
    if min(indices) < 0 or max(indices) >= n_states:
        raise IndexError("POINT_INDICES contains an invalid 2924 point")
    print("State selection:", selection_description)
    print("Selected cache indices:", indices)
    print(f"Selected scan-{LOOP_SCAN} point numbers:", loop_point_indices[indices].tolist())
    reference = h["saturated_hologram"][()]
    painted_detector_mask = prepare_painted_detector_mask(reference, MASK_PIXEL_PNG)
    reference_mask = painted_cache_mask(h, None, painted_detector_mask)
    cached_support_shape = h["saturated_hologram"].shape
    loop_current = h["loop_current_A"][()]
    images = [reference]
    masks = [reference_mask]
    for index in indices:
        images.append(h["loop_holograms"][index])
        masks.append(painted_cache_mask(h, index, painted_detector_mask))
holograms = np.stack(images)
import scipy
mask_pixel = np.stack(masks).astype(bool)  # Exact painted pixels; no implicit dilation.
for column in MASK_COLUMNS:
    if not 0 <= column < mask_pixel.shape[-1]:
        raise ValueError(f"Masked column {column} is outside the detector grid")
    mask_pixel[:, :, column] = 1

intensity_factors = np.ones(len(images), dtype=float)
fitted_offsets = np.zeros(len(images), dtype=float)
for observation in range(1, len(images)):
    pair_mask = mask_pixel[0] | mask_pixel[observation]
    factor, offset = fit_reference_intensity(
        holograms[observation], holograms[0], pair_mask,
    )
    intensity_factors[observation] = factor
    fitted_offsets[observation] = offset
    if NORMALIZE_LOOP_INTENSITY:
        holograms[observation] /= factor
holograms += HOLOGRAM_OFFSET
# Match 01: combine the painted detector masks before binning. The retrieval
# kernel marks nonpositive intensities after crop/binning, as in 01.
reference_mask_for_pairs = mask_pixel[0].copy()
for observation in range(1, len(images)):
    mask_pixel[observation] |= reference_mask_for_pairs
# The saturated input uses the same selected-pair detector mask as 01.
mask_pixel[0] = mask_pixel[comparison_observation].copy()
print("Excluded detector-pixel fractions:", mask_pixel.mean(axis=(1, 2)).round(4))
print("Copy2 intensity factors (diagnostic):", intensity_factors.round(5))
print("Applied loop normalization:", NORMALIZE_LOOP_INTENSITY, "input offset:", HOLOGRAM_OFFSET)
print("Fitted correlation intercepts (diagnostic):", fitted_offsets.round(5))
state_labels = ["saturated"] + [f"loop_{loop_point_indices[index]:04d}" for index in indices]
print("Loaded cache:", CACHE, "stack:", holograms.shape)


In [ ]:
%matplotlib qt
# Inspect the actual detector pixels excluded before phase retrieval.
qa_mask = mask_pixel[0] if mask_pixel.ndim == 3 else mask_pixel
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
axes[0].imshow(np.log1p(np.maximum(reference, 0)), cmap="gray", vmin=0, vmax=3)
axes[0].contour(qa_mask, levels=[0.5], colors="red", linewidths=0.5)
axes[0].set_title(f"{SATURATED_SCAN} hologram with excluded-pixel contour")
axes[1].imshow(qa_mask, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("mask_pixel: white = excluded")
plt.tight_layout()

In [ ]:


temp=fth.reconstruct(holograms[0])
#temp[:,1024]=0
cropl=FILTER_EDGE_PIXELS
temp[:cropl,:]=0
temp[-cropl:,:]=0
temp[:,:cropl]=0
temp[:,-cropl:]=0
temp[:cropl,:]=0
temp=np.real(fth.reconstructCDI(temp))

# filter hologram

In [ ]:
temp = np.array(holograms, dtype=complex, copy=True)
for observation in range(holograms.shape[0]):
    temp[observation] = fth.reconstruct(holograms[observation])

# Suppress the selected object-space edge pixels.
edge = int(FILTER_EDGE_PIXELS)
if edge > 0:
    temp[:, :edge, :] = 0
    temp[:, -edge:, :] = 0
    temp[:, :, :edge] = 0
    temp[:, :, -edge:] = 0

# Optional true crop: this reduces the recalculated hologram dimensions.
if FILTER_TEMP_CROP_PIXELS:
    temp = crop_array_edges(temp, int(FILTER_TEMP_CROP_PIXELS))

holograms = np.real(fth.reconstructCDI(temp))
del temp

# Load the user's current support before adapting every array to the new grid.
support_adjustment = None
support_transfer_source = None
if SUPPORT_FROM_01 is None:
    raise ValueError("Set SUPPORT_FROM_01 to the result saved by 01 with the PNG support adjustment")
if not Path(SUPPORT_FROM_01).is_file():
    raise FileNotFoundError(f"Run 01 and save its result first: {SUPPORT_FROM_01}")
with h5py.File(SUPPORT_FROM_01, "r") as support_result:
    required = ("support_before_adjustment", "support_after_adjustment")
    if any(key not in support_result for key in required):
        raise ValueError(
            f"{SUPPORT_FROM_01} predates support-adjustment saving. "
            "Set SUPPORT_FROM_01 to the result you saved in 01 after choosing "
            "erosion/dilation, or rerun its save cell. No support settings were guessed."
        )
    saved_config = json.loads(support_result.attrs["preprocess_config_json"])
    if any(saved_config.get(key) != cache_config.get(key)
           for key in ("saturated_scan", "loop_scan", "detector_center", "crop_raw_pixels", "binning")):
        raise ValueError("01 used a different scan/support cache")
    saved_shift = tuple(support_result.attrs.get("support_input_shift", (0, 0)))
    support_used = support_result["support_after_adjustment"][()].astype(bool)
    support_adjustment = json.loads(support_result.attrs["support_adjustment_json"])
if support_used.shape != cached_support_shape or not support_used.any():
    raise ValueError("Invalid adjusted support in the 01 result")
support_transfer_source = str(SUPPORT_FROM_01)
print("Using the exact adjusted support saved by 01:", SUPPORT_FROM_01)
print("Support adjustment:", support_adjustment, "; included shift:", saved_shift)
# Do not fill holes or reapply morphology/shift to this transferred mask.
support_input_from_01 = support_used.copy() if support_transfer_source is not None else None

holograms, mask_pixel, support_used, shape_changes = adapt_retrieval_array_shapes(
    holograms, mask_pixel, support_used,
)
for change in shape_changes:
    print("Automatic shape adjustment:", change)
if not shape_changes:
    print("Holograms, mask_pixel, and supportmask already share", holograms.shape[-2:])
print("Filtered retrieval inputs:", holograms.shape, mask_pixel.shape, support_used.shape)
planned_retrieval_shape = tuple(
    (np.asarray(holograms.shape[-2:]) - 2 * RETRIEVAL_CROP) // RETRIEVAL_BINNING
)
if min(planned_retrieval_shape) < 1:
    raise ValueError("FILTER_TEMP_CROP_PIXELS and RETRIEVAL_CROP remove the complete image")
print("Planned shape after the later retrieval crop/bin:", planned_retrieval_shape)

# Review the loaded inputs and effective support before starting retrieval.
def review_json_value(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, slice):
        return [value.start, value.stop, value.step]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Cannot display parameter of type {type(value).__name__}")

loaded_parameter_review = {
    "Files and scans": {
        "cache_loaded": str(CACHE),
        "saturated_scan": cache_config.get("saturated_scan"),
        "loop_scan": cache_config.get("loop_scan"),
        "dark_scan": cache_config.get("dark_scan"),
        "raw_root": cache_config.get("raw_root"),
        "raw_prefix": cache_config.get("prefix"),
        "painted_support_png": support_adjustment.get("support_png", cache_config.get("support_png")),
        "detector_mask_png": cache_config.get("detector_mask_png"),
        "detector_mask_png": str(MASK_PIXEL_PNG),
        "support_result_from_01": support_transfer_source,
        "material_mask_file": MATERIAL_MASK_FILE,
        "relative_thickness_file": RELATIVE_THICKNESS_FILE,
        "result_hdf5": str(OUTPUT),
        "result_png_directory": str(PNG_OUTPUT),
        "focused_difference_png_directory": str(PNG_OUTPUT / "focused_differences"),
    },
    "Support actually loaded": {
        "central_circle_radius": support_adjustment.get("center_radius", "not recorded"),
        "source": "PNG support adjusted and saved by 01",
        "erosion_from_01": (support_adjustment.get("erosion", "not recorded")
                            if support_adjustment is not None else "not inherited"),
        "dilation_from_01": (support_adjustment.get("dilation", "not recorded")
                             if support_adjustment is not None else "not inherited"),
        "adjustment_order": (support_adjustment.get("order", "not recorded")
                              if support_adjustment is not None else "not inherited"),
        "adjustment_pixel_grid": (support_adjustment.get("grid", "not recorded")
                                   if support_adjustment is not None else "not inherited"),
        "morphology_reapplied_in_02": False,
        "holes_filled_in_02": support_transfer_source is None,
        "shift_inherited_from_01": saved_shift if support_transfer_source is not None else None,
        "configured_02_shift": SUPPORT_SHIFT,
        "configured_02_shift_used": False,
        "current_shape_before_retrieval": support_used.shape,
        "support_pixel_count": int(np.count_nonzero(support_used)),
    },
    "Actual state selection and geometry": {
        "selection": selection_description,
        "selected_scan_point_numbers": loop_point_indices[indices],
        "selected_cache_positions": indices,
        "selected_currents_A": loop_current[indices],
        "comparison_scan_point": MASK_REFERENCE_POINT_NUMBER,
        "photon_energy_eV": ENERGY_EV,
        "hologram_stack_shape": holograms.shape,
        "detector_mask_shape": mask_pixel.shape,
        "excluded_detector_fraction_per_observation": np.mean(mask_pixel != 0, axis=(-2, -1)),
        "additional_detector_mask_dilation_in_02": 3,
        "additional_excluded_columns": MASK_COLUMNS,
        "planned_retrieval_shape": planned_retrieval_shape,
        "automatic_shape_changes": shape_changes,
        "fitted_intensity_factors": intensity_factors,
        "loop_intensity_normalization_applied": NORMALIZE_LOOP_INTENSITY,
        "hologram_offset_applied": HOLOGRAM_OFFSET,
    },
    "Preprocessing settings recorded in cache (including photon threshold and detector-mask morphology)": cache_config,
    "Notebook 02 settings (focus values are initial values; PNG export reads live sliders)": {
        name: globals()[name] for name in ('SATURATED_SCAN', 'LOOP_SCAN', 'OUTPUT_DIR', 'CACHE', 'SATURATED_SIGN', 'SUPPORT_SHIFT', 'POINT_INDICES', 'STATE_STRIDE', 'FIRST_N_STATES', 'MASK_REFERENCE_POINT_NUMBER', 'ALWAYS_INCLUDE_COMPARISON_STATE', 'PREVIEW_STATES', 'RETRIEVAL_CROP', 'RETRIEVAL_BINNING', 'NORMALIZE_LOOP_INTENSITY', 'HOLOGRAM_OFFSET', 'FILTER_EDGE_PIXELS', 'FILTER_TEMP_CROP_PIXELS', 'SUPPORT_FROM_01', 'DISPLAY_PROP_UM', 'DISPLAY_PHASE_RAD', 'DISPLAY_SETUP_BASE', 'DIAGNOSTIC_ROI_OVERRIDE', 'DIAGNOSTIC_ROI_PADDING', 'PHYSICAL_PROJECTION_OBJECT_ROI', 'PHYSICAL_PHASE_REFERENCE', 'RUN_FOCUS_WIDGETS', 'LIVE_RECONSTRUCTION', 'LIVE_OBSERVATIONS', 'LIVE_EVERY_PROJECTION', 'LIVE_MIN_SECONDS', 'MODES', 'CONSTRAIN_OTHER_MODES_COMMON', 'OTHER_MODES_COMMON_RELAXATION', 'USE_PARTIAL_COHERENCE', 'WARMUP_POLICY', 'WARMUP_START_FROM_REFERENCE', 'REFERENCE_WARMUP_MODE', 'REFERENCE_WARMUP_NIT', 'OTHER_WARMUP_MODE', 'OTHER_WARMUP_NIT', 'USE_COMMON_COHERENCE_KERNEL', 'SHARED_COHERENCE_UPDATE', 'WARMUP_CALIBRATE_SHARED_GAMMA', 'OBSERVATION_WORKERS', 'FFT_WORKERS', 'COHERENCE_ROUND_ITERATIONS', 'COHERENCE_KERNEL_SCOPE', 'PRESERVE_WARMUP_MASKED_INTENSITY', 'FREEZE_SATURATED_FIELD', 'FIX_COHERENCE_AFTER_SATURATED', 'MODE_INITIALIZATION', 'MODE_INITIALIZATION_SEED', 'STARTIMAGE_RADIAL_NORMALIZATION', 'MODE_SUPPORT_CENTER', 'THICKNESS_APERTURE_MODE', 'MATERIAL_MASK_FILE', 'RELATIVE_THICKNESS_FILE', 'FIT_MATERIAL_THICKNESS', 'ZERO_THICKNESS_OUTSIDE_SUPPORT', 'OUTPUT', 'PNG_OUTPUT', 'RUN_POISSON_REFINEMENT', 'POISSON_STEPS', 'POISSON_LEARNING_RATE', 'REFRESH_MASK_AFTER_OUTER_LOOPS', 'COHERENT_REFRESH_ITERATIONS', 'COHERENT_REFRESH_MODE', 'PHASE_PROPAGATION_SIGN', 'COBALT_THICKNESS_NM', 'NORMALIZE_MAGNETIZATION_TO_UNIT_RANGE', 'LOOP_EXCLUDE_POINT_NUMBERS', 'RETRIEVAL_POINT_NUMBERS', 'MASK_COLUMNS')
    },
}
print("\n================ NOTEBOOK 02: LOADED PARAMETER REVIEW ================")
print("null = not supplied/disabled. PNG paths above are provenance; masks are loaded from the selected cache/result.")
for section, values in loaded_parameter_review.items():
    print(f"\n{section}")
    print(json.dumps(values, indent=2, default=review_json_value))
print("\n================ END OF LOADED PARAMETER REVIEW =====================")


In [ ]:
plt.close("all")
fig,ax=plt.subplots()
ax.imshow(holograms[0], vmin=0, vmax=10)
fig,ax=plt.subplots()
ax.imshow(np.abs(fth.reconstruct(holograms[0])), vmin=0, vmax=0.01)

## Joint physical retrieval

CPU states run in parallel (`OBSERVATION_WORKERS=2`); each uses `FFT_WORKERS=1`.
The reference coherent warmup runs first to seed the other states. Every state
finishes coherent warmup before partial coherence, with its masked intensities
preserved. Physical projections run after complete observation rounds.

With partial coherence enabled, choose `SHARED_COHERENCE_UPDATE`:
- `"average"` (default): each state refines its own copy of the same round-start
  gamma; normalize the kernels and average them after the round. Equal weights
  are used unless `observation_weights` is supplied.
- `"pooled"`: keep gamma fixed during the parallel field updates, then fit one
  gamma against all states together using `COHERENCE_ROUND_ITERATIONS` RL steps.
- `"sequential"`: retain the original state-to-state gamma updates; joint
  partial-coherence updates are serial, while coherent warmup can run in parallel.

Before parallel partial-coherence warmup, `WARMUP_CALIBRATE_SHARED_GAMMA=True`
fits gamma on saturation and seeds the loop states with it. All states still
contribute to the warmup gamma average and subsequent round averages.

Frozen saturation contributes to gamma fitting without changing its field.
With two modes, gamma has one spatial kernel per mode, shared across states.
A GPU backend automatically uses one observation worker. Set
`USE_COMMON_COHERENCE_KERNEL=False` for independent per-state kernels.

`PHYSICAL_PROJECTION_OBJECT_ROI=True` fits only positive-thickness pixels in the
object aperture, with full-size component maps returned for compatibility. The
exterior is preserved, including reference holes; it is no longer forced to the
common vacuum field at full strength. Focus FFTs and the separate common
secondary-mode constraint still use the full field. The displayed ROI is derived
from the actual retrieval support, so cropping, binning and recentering are
already accounted for. A manual retrieval-grid override remains available.

The common secondary mode uses a phase-aligned complex-field average, avoiding
wrapped-log phase artifacts. `PHYSICAL_PHASE_REFERENCE=True` places primary-mode
log phases on the nearest branch to the first state in each energy/beam group
(saturation here). This preserves the input wavefields and assumes relative
state phases differ by less than pi; it does not unwrap arbitrary multi-turn
phase differences. The error diagnostics trace primary and total intensity in
outer explicitly masked detector pixels before and after the projections.


In [ ]:
if any(isinstance(loop, bool) or not isinstance(loop, int) or loop < 1
       for loop in REFRESH_MASK_AFTER_OUTER_LOOPS):
    raise ValueError("REFRESH_MASK_AFTER_OUTER_LOOPS must contain positive completed loop numbers")

plt.close("all")

from library import phase_retrieval_universal as universal
from library import phase_retrieval_core_unified as unified
from library.retrieval_progress import retrieval_progress
from PIL import Image
# Optional object-space inputs. A nonzero material mask means sample material;
# zero marks known vacuum regions, including reference holes.
material_mask = None
if MATERIAL_MASK_FILE is not None:
    material_path = Path(MATERIAL_MASK_FILE)
    if material_path.suffix.lower() == ".npy":
        material_mask = np.load(material_path)
    else:
        material_mask = np.asarray(Image.open(material_path).convert("L"))

relative_thickness = None
if RELATIVE_THICKNESS_FILE is not None:
    thickness_path = Path(RELATIVE_THICKNESS_FILE)
    if thickness_path.suffix.lower() == ".npy":
        relative_thickness = np.load(thickness_path)
    else:
        relative_thickness = np.asarray(Image.open(thickness_path).convert("L"))

# Supplied object maps may still use the pre-filter source or retrieval grid.
# Keep their object-space pixel scale and center-crop/pad to the nearest valid grid.
source_grid_shape = tuple(holograms.shape[-2:])
retrieval_grid_shape = tuple(
    (np.asarray(source_grid_shape) - 2 * RETRIEVAL_CROP) // RETRIEVAL_BINNING
)
if material_mask is not None and material_mask.shape not in {source_grid_shape, retrieval_grid_shape}:
    material_target_shape = min(
        (source_grid_shape, retrieval_grid_shape),
        key=lambda shape: np.sum(np.abs(np.asarray(shape) - material_mask.shape)),
    )
    print("Automatic shape adjustment: material_mask", material_mask.shape,
          "->", material_target_shape, "(center crop/pad)")
    material_mask = center_crop_or_pad(material_mask, material_target_shape, fill_value=0)
if (relative_thickness is not None
        and relative_thickness.shape not in {source_grid_shape, retrieval_grid_shape}):
    thickness_target_shape = min(
        (source_grid_shape, retrieval_grid_shape),
        key=lambda shape: np.sum(np.abs(np.asarray(shape) - relative_thickness.shape)),
    )
    print("Automatic shape adjustment: relative_thickness", relative_thickness.shape,
          "->", thickness_target_shape, "(center crop/pad)")
    relative_thickness = center_crop_or_pad(
        relative_thickness, thickness_target_shape, fill_value=0,
    )
# Translation of an object support changes only Fourier phase, so detector
# exclusion pixels remain fixed. Move supplied object maps with the support.
support_used, support_source_shift = recenter_source_support(
    support_used, RETRIEVAL_CROP, RETRIEVAL_BINNING,
)
print("Support shift on cached object grid:", support_source_shift)
if any(support_source_shift):
    from scipy.ndimage import shift as shift_object_map
    source_shape = np.asarray(support_used.shape)
    retrieval_shape = (source_shape - 2 * RETRIEVAL_CROP) // RETRIEVAL_BINNING
    retrieval_shift = np.rint(
        np.asarray(support_source_shift) * (source_shape - 2 * RETRIEVAL_CROP)
        / source_shape,
    ).astype(int)
    if material_mask is not None:
        map_shift = (support_source_shift if material_mask.shape == tuple(source_shape)
                     else retrieval_shift if material_mask.shape == tuple(retrieval_shape)
                     else None)
        if map_shift is None:
            raise ValueError("Material mask must be on the cached or retrieval object grid")
        material_mask = shift_object_map(material_mask, map_shift, order=0,
                                         mode="constant", cval=0, prefilter=False)
    if relative_thickness is not None:
        map_shift = (support_source_shift if relative_thickness.shape == tuple(source_shape)
                     else retrieval_shift if relative_thickness.shape == tuple(retrieval_shape)
                     else None)
        if map_shift is None:
            raise ValueError("Thickness map must be on the cached or retrieval object grid")
        relative_thickness = shift_object_map(relative_thickness, map_shift, order=1,
                                              mode="constant", cval=0, prefilter=False)
if THICKNESS_APERTURE_MODE not in {"none", "support", "fixed"}:
    raise ValueError("THICKNESS_APERTURE_MODE must be none, support, or fixed")
if THICKNESS_APERTURE_MODE != "none":
    if material_mask is not None or relative_thickness is not None:
        raise ValueError("Choose either a thickness aperture mode or a supplied material map")
    thickness_aperture = universal.largest_support_component(support_used)
    print("Object aperture pixels:", int(thickness_aperture.sum()))
    plt.figure(figsize=(4, 4))
    plt.imshow(thickness_aperture, cmap="gray")
    plt.title("Binary thickness aperture from largest support component")
    plt.show()
    if THICKNESS_APERTURE_MODE == "support":
        material_mask = thickness_aperture
    else:
        relative_thickness = thickness_aperture
fit_thickness = FIT_MATERIAL_THICKNESS or THICKNESS_APERTURE_MODE == "support"
if THICKNESS_APERTURE_MODE == "fixed" and FIT_MATERIAL_THICKNESS:
    raise ValueError("Fixed binary thickness requires FIT_MATERIAL_THICKNESS=False")
if not isinstance(MODES, (list, tuple)) or not MODES:
    raise ValueError("MODES must be a nonempty list of positive support factors")
INNER_NIT = [100,100]
if WARMUP_POLICY not in {"same_for_all", "reference_and_others"}:
    raise ValueError("WARMUP_POLICY must be 'same_for_all' or 'reference_and_others'")

# Repeat the chosen coherent recipes with Richardson-Lucy updates when partial
# coherence is enabled. The driver completes the coherent pass for ALL states
# before the partial pass, and preserves each coherent masked-pixel intensity.
def add_partial_coherence_pass(modes, iterations):
    if not USE_PARTIAL_COHERENCE:
        return list(modes), list(iterations), [0] * len(modes), [10**9] * len(modes)
    return (list(modes) + list(modes), list(iterations) + list(iterations),
            [0] * len(modes) + [50] * len(modes),
            [10**9] * len(modes) + [20] * len(modes))

reference_modes, reference_iterations, reference_rl_it, reference_rl_freq = (
    add_partial_coherence_pass(REFERENCE_WARMUP_MODE, REFERENCE_WARMUP_NIT)
)
other_modes, other_iterations, other_rl_it, other_rl_freq = (
    add_partial_coherence_pass(OTHER_WARMUP_MODE, OTHER_WARMUP_NIT)
)
print("Warmup policy:", WARMUP_POLICY)
print("  saturated:", list(zip(reference_modes, reference_iterations)))
print("  loop states:", (list(zip(reference_modes, reference_iterations))
      if WARMUP_POLICY == "same_for_all" else list(zip(other_modes, other_iterations))))
recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    material_mask=material_mask, material_thickness=relative_thickness,
    fit_material_thickness=fit_thickness,
    zero_thickness_outside_support=ZERO_THICKNESS_OUTSIDE_SUPPORT,
    projection_model="physical_factorized",
    physical_projection_object_roi=PHYSICAL_PROJECTION_OBJECT_ROI,
    physical_phase_reference=PHYSICAL_PHASE_REFERENCE,
    projection_diagnostic_observation=comparison_observation,
    projection_focus_prop_um=DISPLAY_PROP_UM,
    projection_focus_phase_rad=DISPLAY_PHASE_RAD,
    projection_focus_setup=DISPLAY_SETUP,
    modes=MODES, mode_initialization=MODE_INITIALIZATION,
    constrain_nonphysical_modes_common=CONSTRAIN_OTHER_MODES_COMMON,
    nonphysical_modes_common_relaxation=OTHER_MODES_COMMON_RELAXATION,
    mode_initialization_seed=MODE_INITIALIZATION_SEED,
    mode_support_center=MODE_SUPPORT_CENTER,
    partial_coherence=USE_PARTIAL_COHERENCE,
    coherence_kernel_scope=COHERENCE_KERNEL_SCOPE,
    shared_coherence_update=(SHARED_COHERENCE_UPDATE
                             if USE_COMMON_COHERENCE_KERNEL and not FIX_COHERENCE_AFTER_SATURATED
                             else "sequential"),
    observation_workers=OBSERVATION_WORKERS, fft_workers=FFT_WORKERS,
    warmup_calibrate_shared_gamma=WARMUP_CALIBRATE_SHARED_GAMMA,
    coherence_round_iterations=COHERENCE_ROUND_ITERATIONS,
    preserve_warmup_masked_intensity=PRESERVE_WARMUP_MASKED_INTENSITY,
    freeze_coherence_after_reference=(USE_COMMON_COHERENCE_KERNEL and FIX_COHERENCE_AFTER_SATURATED),
    final_fourier_constraint=not USE_PARTIAL_COHERENCE,
    recenter_modal_supports=True,
    saturated_states={"saturated": SATURATED_SIGN},
    freeze_saturated_fields=FREEZE_SATURATED_FIELD,
    energy_values=np.array([ENERGY_EV]),
    # Shared schedule is used by WARMUP_POLICY="same_for_all". The explicit
    # role schedules below are used by "reference_and_others".
    warmup_mode=reference_modes, warmup_Nit=reference_iterations,
    warmup_RL_it=reference_rl_it, warmup_RL_freq=reference_rl_freq,
    warmup_reference_mode=(reference_modes if WARMUP_POLICY == "reference_and_others" else None),
    warmup_reference_Nit=(reference_iterations if WARMUP_POLICY == "reference_and_others" else None),
    warmup_reference_RL_it=(reference_rl_it if WARMUP_POLICY == "reference_and_others" else None),
    warmup_reference_RL_freq=(reference_rl_freq if WARMUP_POLICY == "reference_and_others" else None),
    warmup_other_mode=(other_modes if WARMUP_POLICY == "reference_and_others" else None),
    warmup_other_Nit=(other_iterations if WARMUP_POLICY == "reference_and_others" else None),
    warmup_other_RL_it=(other_rl_it if WARMUP_POLICY == "reference_and_others" else None),
    warmup_other_RL_freq=(other_rl_freq if WARMUP_POLICY == "reference_and_others" else None),
    # Keep warmup identical to 01: no TV/support penalty during these stages.
    warmup_alpha_zero=[0.4] * len(reference_modes),
    warmup_alpha_mode=["linear_to_0"] * len(reference_modes),
    warmup_TV_freq=[5] * len(reference_modes),
    warmup_reference_alpha_zero=([0.4] * len(reference_modes)
                                 if WARMUP_POLICY == "reference_and_others" else None),
    warmup_reference_alpha_mode=(["linear_to_0"] * len(reference_modes)
                                 if WARMUP_POLICY == "reference_and_others" else None),
    warmup_reference_TV_freq=([5] * len(reference_modes)
                              if WARMUP_POLICY == "reference_and_others" else None),
    warmup_other_alpha_zero=([0.4] * len(other_modes)
                             if WARMUP_POLICY == "reference_and_others" else None),
    warmup_other_alpha_mode=(["linear_to_0"] * len(other_modes)
                             if WARMUP_POLICY == "reference_and_others" else None),
    warmup_other_TV_freq=([5] * len(other_modes)
                          if WARMUP_POLICY == "reference_and_others" else None),
    warmup_start_from_first=WARMUP_START_FROM_REFERENCE,
    warmup_reference_observation=warmup_reference_observation,
    startimage_scale_fit="through_origin",
    startimage_radial_normalization=STARTIMAGE_RADIAL_NORMALIZATION,
    coherent_refresh_rounds=[loop + 1 for loop in REFRESH_MASK_AFTER_OUTER_LOOPS],
    coherent_refresh_iterations=COHERENT_REFRESH_ITERATIONS,
    coherent_refresh_mode=COHERENT_REFRESH_MODE,
    warmup_seed_scale_fit="linear",
    warmup_seeded_stage_indices=None,  # Explicit role recipes replace stage-index selection.
    warmup_beta_mode=["arctan" if mode == "HAPRE" else "const" for mode in reference_modes],
    warmup_reference_beta_mode=(["arctan" if mode == "HAPRE" else "const"
                                 for mode in reference_modes]
                                if WARMUP_POLICY == "reference_and_others" else None),
    warmup_other_beta_mode=(["const"] * len(other_modes)
                            if WARMUP_POLICY == "reference_and_others" else None),
    beta_mode="const",
    beta_zero=0.5, average_img=10,
    inner_mode=["ER","ER"], inner_Nit=INNER_NIT,
    RL_it=[50,50] if USE_PARTIAL_COHERENCE else 0,
    RL_freq=20,  # The shared-reference option freezes gamma explicitly in the driver.
    outer_iterations=4, physical_iterations=3,
    projection_every=len(state_labels), projection_relaxation=0.2,
    final_projection_relaxation=0.0,  # Skip the final physical projection.
    projection_constraints_inside_support_only=True,
    zero_magnetization_outside_support=True,
    hologram_intensity_cutoff_vmin=-1,
    shuffle_observations=True, random_seed=0,
    crop=RETRIEVAL_CROP, binning=RETRIEVAL_BINNING, roi=None,
    alpha_zero= 0.4,
    alpha_mode= 'linear_to_0',
    TV_freq= 5,
)
# No start_fields argument: for MODES=[1], initialize from the inverse FFT of the supportmask
# and scale to each measured amplitude. Warm saturated scan 2923 first,
# Reference seeding is optional; with WARMUP_START_FROM_REFERENCE=False all starts are independent.
# For MODES=[1, 2], MODE_INITIALIZATION controls the modal start images.
# Optional radial normalization then sets sum(abs(startimage)**2 over modes)
# to the hologram mean in each 1-pixel detector annulus, preserving phases
# and relative mode weights. Only mask_pixel==0, finite nonnegative samples
# enter the mean (including zeros); empty rings interpolate, with nearest
# extrapolation at the ends. The center/grid follow retrieval crop and binning.
# Applied also to reference-seeded states, but not explicitly supplied start_fields.
# Exact field zeros use phase zero in mode 1. This is only an initial guess:
# it neither freezes masked pixels nor enforces radial symmetry during retrieval.
# Set STARTIMAGE_RADIAL_NORMALIZATION=False to restore the previous starts.

plt.close("all")
%matplotlib inline
# Always display the resolved recipe before computation.
universal.print_universal_workflow(recipe, state_labels=state_labels)
recipe_figure = universal.plot_universal_workflow(recipe, state_labels=state_labels)
plt.show()

from library.retrieval_live import LiveReconstruction
live_view = (LiveReconstruction(
    observations=([0, comparison_observation] if LIVE_OBSERVATIONS is None else LIVE_OBSERVATIONS),
    every=LIVE_EVERY_PROJECTION, min_seconds=LIVE_MIN_SECONDS,
) if LIVE_RECONSTRUCTION else None)

with retrieval_progress(recipe, len(state_labels), kind="universal"):
    fields, warmup, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
        holograms, mask_pixel, support_used,
        state_labels=state_labels,
        energy_labels=[ENERGY_EV]*len(state_labels),
        polarization_coefficients=[1]*len(state_labels),
        illumination_labels=["beam0"]*len(state_labels),
        universal_recipe=recipe,
        # Use the same coherent/multimode Fourier kernel as notebook 01.
        phase_retrieval_kernel=unified.PhaseRtrv_core,
        progress_callback=live_view,
    )

if list(components["state_names"]) != state_labels:
    raise RuntimeError("The physical fit did not return one state per selected field point")
print("Measured execution:", errors.get("execution_summary", {}))
print("Warmup pass timings:", errors.get("warmup_passes", []))
from library.phase_retrieval_geometry import support_bounding_roi
DIAGNOSTIC_ROI = (support_bounding_roi(components["supportmask_used"], DIAGNOSTIC_ROI_PADDING)
                  if DIAGNOSTIC_ROI_OVERRIDE is None else DIAGNOSTIC_ROI_OVERRIDE)
print("Display ROI on retrieval grid:", DIAGNOSTIC_ROI)
print("Physical fit pixels:", components.get("physical_projection_fit_pixels"),
      "of", components.get("physical_projection_full_pixels"))
print("Modal support shifts on retrieval grid:", components["mode_support_shifts"])
print("Frozen saturated observations:", components["frozen_saturated_observations"])
print("Partial coherence fitted:", components["partial_coherence"],
      "kernel scope:", components["coherence_kernel_scope"])
print("Retrieved:", fields.shape, "modal support factors:",
      components["modes"], "physical model modes:",
      components.get("physical_model_modes", [1]), "magnetic state maps:",
      np.asarray(components["magnetization"]).shape[0],
      "fit residual RMS:", components["fit_residual_rms"])
print("Model identifiable flag:", components["identifiable"],
      "(absolute charge and magnetic scale require additional constraints)")

# Array mode 0 has support factor 1; only this mode enters the physical projection.
primary_fields = fields[:, 0] if fields.ndim == 4 else fields
primary_warmup = warmup[:, 0] if warmup.ndim == 4 else warmup
if fields.ndim == 4 and components.get("physical_model_modes") != [1]:
    raise RuntimeError("Expected physical projection only on support-factor-1 mode")

## Optional final statistical refinement

The current retrieval is kept in `fields`; `warmup` is still the end of the entire
warmup, including partial coherence. The separate `refined_fields` result uses a
Poisson-shaped loss in camera units, not FIVS or a calibrated photon likelihood.
Thresholding, dark subtraction and photon spreading violate the independent
Poisson model, so this is an experimental noise-aware objective.

This last step avoids imposing every noisy detector amplitude exactly. Gamma,
saturated observations and secondary modes are fixed; only each remaining
primary field is polished inside its support. Physical/magnetic components still
describe the baseline, not the refined fields. No physical projection or exact
amplitude replacement follows this refinement. Valid measured zeros contribute;
negative/nonfinite intensities and explicitly masked pixels are excluded. Filled
beamstop values are not treated as observations. Existing warmup fills are unchanged.

Backtracking ensures descent after initial support enforcement, which can itself
increase the loss; diagnostics report the original baseline separately. Lower
training loss alone does not prove better reconstruction: compare independent
frame subsets before choosing a final iteration count.


In [ ]:
from library import poisson_refinement as poisson
from library import phase_retrieval_geometry as retrieval_geometry

refined_fields = None
poisson_records = []
poisson_stage_deviance = None
if RUN_POISSON_REFINEMENT:
    poisson_measured, poisson_mask, _, _, _ = retrieval_geometry.prepare(
        holograms, mask_pixel, support_used, dict(recipe, Nmodes=1),
    )
    if poisson_measured.shape != fields.shape[:1] + fields.shape[-2:]:
        raise ValueError("Refinement detector grid differs from retrieval output")
    if recipe["hologram_intensity_cutoff_vmin"] != -1:
        raise ValueError("Refinement requires unshifted detector intensities (cutoff_vmin=-1)")
    poisson_mask = np.broadcast_to(poisson_mask, poisson_measured.shape)
    poisson_support = components["modal_supportmask_used"]
    refined_fields = fields.copy()
    frozen_observations = set(components.get("frozen_saturated_observations", []))

    def observation_gamma(observation):
        if not components.get("partial_coherence", False):
            return None
        if components["coherence_kernel_scope"] == "shared":
            return components["coherence_kernel"]
        return components["coherence_kernels"][observation]

    from tqdm.auto import tqdm
    for observation in tqdm(range(len(fields)), desc="Poisson primary-field refinement"):
        if observation in frozen_observations:
            poisson_records.append(dict(status="frozen", loss=[], step_sizes=[]))
            continue
        result = poisson.refine_poisson(
            fields[observation], poisson_measured[observation], poisson_support,
            poisson_mask[observation], gamma=observation_gamma(observation),
            steps=POISSON_STEPS, learning_rate=POISSON_LEARNING_RATE, refine_modes=[0],
        )
        refined_fields[observation] = result.fields
        poisson_records.append(dict(status=result.status, loss=result.loss,
                                    step_sizes=result.step_sizes,
                                    baseline_loss=result.baseline_loss,
                                    valid_pixels=result.valid_pixels, zero_pixels=result.zero_pixels))
    poisson_stage_deviance = np.array([
        [poisson.poisson_deviance(
            poisson.predicted_intensity(stack[observation], observation_gamma(observation)),
            poisson_measured[observation], poisson_mask[observation],
        ) for stack in (warmup, fields, refined_fields)]
        for observation in range(len(fields))
    ])
    print("Mean deviance in camera units (not calibrated chi-square): warmup / final / refined")
    print("All three are evaluated with the SAME final gamma.")
    for label, scores, record in zip(state_labels, poisson_stage_deviance, poisson_records):
        print(label, np.round(scores, 6), record["status"])


## Organized phase-retrieval diagnostics

Run these cells after the retrieval cell. They use the same comparison state, focus settings, and mode-0 physical field throughout.


In [ ]:
plt.close("all")

# Complete sampled error history, followed by the physical-projection residuals.
def concatenate_error_history(records, key):
    values = []
    boundaries = []
    labels = []
    cursor = 0
    for record in records:
        sample = np.asarray(record.get(key, []), dtype=float).ravel()
        sample = sample[np.isfinite(sample)]
        if sample.size == 0:
            continue
        values.append(sample)
        cursor += sample.size
        boundaries.append(cursor)
        labels.append(
            f"{record.get('stage', '?')} | obs {record.get('observation', '?')} | "
            f"{record.get('mode', '?')} × {record.get('Nit', '?')}"
        )
    return (np.concatenate(values) if values else np.array([]),
            boundaries, labels)

diffraction_error, error_boundaries, error_labels = concatenate_error_history(
    errors.get("observation_steps", []), "error",
)
support_error, _, _ = concatenate_error_history(
    errors.get("observation_steps", []), "support_error",
)
projection_history = errors.get("projection_steps", [])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
if diffraction_error.size:
    axes[0].semilogy(np.maximum(np.abs(diffraction_error), 1e-30),
                     label="diffraction error", linewidth=1)
if support_error.size:
    axes[0].semilogy(np.maximum(np.abs(support_error), 1e-30),
                     label="support error", linewidth=1, alpha=0.8)
for boundary in error_boundaries[:-1]:
    axes[0].axvline(boundary, color="0.75", linewidth=0.5)
axes[0].set(title="All phase-retrieval errors", xlabel="sampled algorithm step",
            ylabel="error")
axes[0].legend()
axes[0].grid(alpha=0.2)
if projection_history:
    projection_x = [item["completed_update"] for item in projection_history]
    projection_error = [item["fit_residual_rms"] for item in projection_history]
    axes[1].semilogy(projection_x, np.maximum(projection_error, 1e-30), "o-")
axes[1].set(title="Physical-projection residual",
            xlabel="completed hologram updates", ylabel="fit residual RMS")
axes[1].grid(alpha=0.2)
plt.tight_layout()
print("Error-stage order:")
for index, label in enumerate(error_labels):
    print(f"  {index:02d}: {label}")

# These are free detector pixels, not frozen targets during coherent retrieval.
trace = errors.get("detector_constraint_diagnostics", [])
if trace:
    stages = [item["stage"] for item in trace]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
    for key, label in [("primary_outer_masked_mean", "Primary mode"),
                       ("total_outer_masked_mean", "Total modes")]:
        values = [np.nan if item[key] is None else item[key] for item in trace]
        axes[0].plot(values, "o-", label=label)
    axes[0].set(title="Outer explicit mask: mean inferred intensity", ylabel="Camera intensity")
    axes[0].legend()
    axes[1].plot([item["observed_amplitude_relative_error"] for item in trace], "o-")
    axes[1].set(title="Observed pixels: amplitude mismatch", ylabel="Relative error")
    for ax in axes:
        ax.set_xticks(range(len(stages)), stages, rotation=60, ha="right")
        ax.grid(alpha=.2)
    print("Trace observation:", trace[0]["observation"], "outer masked pixels:", trace[0]["outer_masked_pixels"])
    print("Outer = radius > one quarter of the smaller detector dimension.")
    print("With partial coherence, amplitude mismatch is diagnostic only; it is not the blurred-intensity objective.")
    plt.show()


## Reconstruction images with shared color scales

Export every final state (including saturation) and every incoherent mode as full-frame and ROI PNGs: real, imaginary, amplitude, and phase. Primary-mode differences use **state minus saturation**. All states and modes share one scale per component; real and imaginary share a symmetric scale, phase is fixed to ±π. Difference images share a separate symmetric scale. Limits use the full ensemble, without per-image autoscaling, and are recorded in `display_settings.json`. The comparison preview uses the same limits, including warmup and optional refinement in their calculation. Here `warmup` means after both coherent and partial-coherence warmup when partial coherence is enabled. PNGs use the configured focus and phase rotation; complex arrays remain in the HDF5 output.


In [ ]:
plt.close("all")

fig,ax=plt.subplots(1,2)
for i in range(2):
    ax[i].imshow(np.log10(np.abs(fields[4,i])**2))

In [ ]:
# One display transform for previews and all PNG exports.
from matplotlib.colors import Normalize

def display_object(field):
    return fth.reconstructCDI(
        fth.propagate(field, DISPLAY_PROP_UM * 1e-6, experimental_setup=DISPLAY_SETUP)
    ) * np.exp(1j * DISPLAY_PHASE_RAD)

def mode_stack(stack):
    stack = np.asarray(stack)
    if stack.ndim == 3:
        return stack[:, None]
    if stack.ndim != 4:
        raise ValueError("Expected fields shaped (state, [mode,] row, column)")
    return stack

final_modes = mode_stack(fields)
warmup_modes = mode_stack(warmup)
if final_modes.shape != warmup_modes.shape or final_modes.shape[0] != len(state_labels):
    raise ValueError("Warmup, final fields and state labels must match")
if final_modes[0, 0][DIAGNOSTIC_ROI].size == 0:
    raise ValueError("DIAGNOSTIC_ROI is empty on the reconstruction grid")

# Stream transformed objects to avoid retaining another full complex stack.
signed_max = amplitude_max = difference_max = 0.0
comparison_stacks = [warmup_modes, final_modes]
if refined_fields is not None:
    comparison_stacks.append(mode_stack(refined_fields))
for field_stack in comparison_stacks:
    saturated_object = display_object(field_stack[0, 0])
    for observation in range(field_stack.shape[0]):
        for mode_index in range(field_stack.shape[1]):
            obj = display_object(field_stack[observation, mode_index])
            if not np.all(np.isfinite(obj)):
                raise ValueError(f"Nonfinite reconstruction: {state_labels[observation]}, mode {mode_index}")
            signed_max = max(signed_max, float(np.abs(obj.real).max()), float(np.abs(obj.imag).max()))
            amplitude_max = max(amplitude_max, float(np.abs(obj).max()))
            if mode_index == 0:
                difference = obj - saturated_object
                difference_max = max(difference_max, float(np.abs(difference.real).max()),
                                     float(np.abs(difference.imag).max()))
signed_max = signed_max or 1.0
amplitude_max = amplitude_max or 1.0
difference_max = difference_max or 1.0
image_specs = {
    "real": (np.real, "gray", Normalize(-signed_max, signed_max)),
    "imaginary": (np.imag, "gray", Normalize(-signed_max, signed_max)),
    "amplitude": (np.abs, "magma", Normalize(0, amplitude_max)),
    "phase": (np.angle, "twilight", Normalize(-np.pi, np.pi)),
}
difference_norm = Normalize(-difference_max, difference_max)
print("Shared real/imaginary limits:", (-signed_max, signed_max))
print("Shared amplitude limits:", (0, amplitude_max))
print("Shared difference limits:", (-difference_max, difference_max))



In [ ]:
plt.close("all")

# Compact warmup/final comparison; one colorbar per component row.
diagnostic_observation = comparison_observation
diagnostic_label = state_labels[diagnostic_observation]
comparison_objects = [
    display_object(primary_warmup[0]),
    display_object(primary_warmup[diagnostic_observation]),
    display_object(primary_fields[0]),
    display_object(primary_fields[diagnostic_observation]),
]
comparison_titles = ["Warmup saturated", f"Warmup {diagnostic_label}",
                     "Final saturated", f"Final {diagnostic_label}"]
if refined_fields is not None:
    primary_refined = refined_fields[:, 0] if refined_fields.ndim == 4 else refined_fields
    comparison_objects.extend([display_object(primary_refined[0]),
                               display_object(primary_refined[diagnostic_observation])])
    comparison_titles.extend(["Refined saturated", f"Refined {diagnostic_label}"])
fig, axes = plt.subplots(3, len(comparison_objects), figsize=(5 * (len(comparison_objects) // 2), 10), layout="constrained")
for row, name in enumerate(("real", "imaginary", "amplitude")):
    component, cmap, norm = image_specs[name]
    for column, (obj, title) in enumerate(zip(comparison_objects, comparison_titles)):
        artist = axes[row, column].imshow(component(obj)[DIAGNOSTIC_ROI],
                                         cmap=cmap, norm=norm, interpolation="nearest")
        axes[row, column].set_title(f"{title}\n{name}", fontsize=9)
        axes[row, column].axis("off")
    fig.colorbar(artist, ax=axes[row].tolist(), shrink=0.8, label=name)
PNG_OUTPUT.mkdir(parents=True, exist_ok=True)
fig.savefig(PNG_OUTPUT / "warmup_final_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

difference_stages = [(0, "Warmup"), (2, "Final")]
if refined_fields is not None:
    difference_stages.append((4, "Refined"))
fig, axes = plt.subplots(2, len(difference_stages), figsize=(5 * len(difference_stages), 8), layout="constrained")
for column, (offset, title) in enumerate(difference_stages):
    difference = comparison_objects[offset + 1] - comparison_objects[offset]
    for row, (component, name) in enumerate(((np.real, "real"), (np.imag, "imaginary"))):
        artist = axes[row, column].imshow(component(difference)[DIAGNOSTIC_ROI],
                                         cmap="gray", norm=difference_norm, interpolation="nearest")
        axes[row, column].set_title(f"{title}: {diagnostic_label} − saturated | {name}")
        axes[row, column].axis("off")
fig.colorbar(artist, ax=axes.ravel().tolist(), shrink=0.8, label="Difference")
fig.savefig(PNG_OUTPUT / "warmup_final_difference.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
%matplotlib qt

# Detector-space checks for all three reconstruction stages.
if refined_fields is not None:
    obs = diagnostic_observation
    measured = poisson_measured[obs]
    valid = poisson.valid_pixels(measured, poisson_mask[obs])
    predictions = [poisson.predicted_intensity(stack[obs], observation_gamma(obs))
                   for stack in (warmup, fields, refined_fields)]
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), layout="constrained")
    images = [np.where(valid, np.maximum(measured, 0), np.nan)] + predictions
    top = max(float(np.nanpercentile(np.log1p(im), 99.8)) for im in images)
    for ax, im, title in zip(axes[0], images, ["Measured", "Warmup", "Final", "Refined"]):
        artist = ax.imshow(np.log1p(im), vmin=0, vmax=top, cmap="magma")
        ax.set_title(title + " | log(1 + intensity)")
    fig.colorbar(artist, ax=axes[0].tolist(), shrink=.7)
    axes[1, 0].imshow(~valid, cmap="gray")
    axes[1, 0].set_title("Excluded pixels (white)")
    residuals = [np.where(valid, pred - measured, np.nan) for pred in predictions]
    limit = max(float(np.nanpercentile(np.abs(r), 99)) for r in residuals) or 1.
    for ax, residual, title in zip(axes[1, 1:], residuals, ["Warmup", "Final", "Refined"]):
        artist = ax.imshow(residual, cmap="RdBu_r", vmin=-limit, vmax=limit)
        ax.set_title(title + " − measured")
    fig.colorbar(artist, ax=axes[1, 1:].tolist(), shrink=.7)
    PNG_OUTPUT.mkdir(parents=True, exist_ok=True)
    fig.savefig(PNG_OUTPUT / "poisson_detector_comparison.png", dpi=150)
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
    for record in poisson_records:
        if len(record["loss"]):
            axes[0].plot(np.asarray(record["loss"]) - record["baseline_loss"], alpha=.4)
    axes[0].axhline(0, color="black", linewidth=.7)
    axes[0].set(title="Loss change from original final field", xlabel="Refinement step", ylabel="Quasi-Poisson loss change")
    for column, label in enumerate(["Warmup", "Final", "Refined"]):
        axes[1].plot(poisson_stage_deviance[:, column], label=label)
    axes[1].set(title="All states, same final gamma", xlabel="Observation", ylabel="Mean deviance (camera units)")
    axes[1].legend()
    yy, xx = np.indices(measured.shape)
    radius = np.hypot(yy - measured.shape[0] // 2, xx - measured.shape[1] // 2).astype(int)
    counts = np.bincount(radius[valid])
    for im, label in zip([np.maximum(measured, 0)] + predictions, ["Measured", "Warmup", "Final", "Refined"]):
        radial = np.bincount(radius[valid], weights=im[valid]) / np.maximum(counts, 1)
        axes[2].semilogy(np.where(counts > 0, np.maximum(radial, 1e-12), np.nan), label=label)
    axes[2].set(title="Radial intensity (valid pixels only)", xlabel="Detector radius (pixels)", ylabel="Mean camera intensity")
    axes[2].legend()
    fig.savefig(PNG_OUTPUT / "poisson_refinement_diagnostics.png", dpi=150)
    plt.show()


Focused difference PNGs are exported in the cell immediately after the focus widgets. Adjust the sliders first, then run that export cell.


### Interactive focusing of the warmup and final differences

The widgets use the same propagation distance, phase, detector geometry, ROI, and subtraction convention as the physical projection diagnostics. `dx=dy=0` initially.


In [ ]:
plt.close("all")
%matplotlib qt

diagnostic_observation = 5#comparison_observation
focus_export_sliders = {}  # Reset handles when rebuilding the widgets.
print(RUN_FOCUS_WIDGETS)
if RUN_FOCUS_WIDGETS:
    import importlib
    ajajas_library = str(HERE / "library")
    if ajajas_library not in sys.path:
        sys.path.insert(0, ajajas_library)
    rec = importlib.import_module("reconstruct_rb")

    if True:
        print("Warmup difference focus")
        warmup_slider_prop, warmup_slider_phase, warmup_slider_dx, warmup_slider_dy = rec.focusCDI(
            primary_warmup[diagnostic_observation], primary_warmup[0],
            DIAGNOSTIC_ROI, mask=1, phase=DISPLAY_PHASE_RAD, dx=0, dy=0,
            prop_dist=DISPLAY_PROP_UM, experimental_setup=DISPLAY_SETUP,
            operation="-", max_prop_dist=10, scale=(1, 99),
        )

        focus_export_sliders["warmup"] = (warmup_slider_prop, warmup_slider_phase, warmup_slider_dx, warmup_slider_dy)

    if False:
        print("Final difference focus")
        final_slider_prop, final_slider_phase, final_slider_dx, final_slider_dy = rec.focusCDI(
            primary_fields[diagnostic_observation], primary_fields[0],
            DIAGNOSTIC_ROI, mask=1, phase=DISPLAY_PHASE_RAD, dx=0, dy=0,
            prop_dist=DISPLAY_PROP_UM, experimental_setup=DISPLAY_SETUP,
            operation="-", max_prop_dist=5, scale=(1, 99),
        )

        focus_export_sliders["final"] = (final_slider_prop, final_slider_phase, final_slider_dx, final_slider_dy)

    if False:
        if refined_fields is not None:
            print("Refined difference focus (fixed final gamma)")
            refined_sliders = rec.focusCDI(
                primary_refined[diagnostic_observation], primary_refined[0],
                DIAGNOSTIC_ROI, mask=1, phase=DISPLAY_PHASE_RAD, dx=0, dy=0,
                prop_dist=DISPLAY_PROP_UM, experimental_setup=DISPLAY_SETUP,
                operation="-", max_prop_dist=5, scale=(1, 99),
            )
            focus_export_sliders["refined"] = refined_sliders


In [ ]:
# Run AFTER adjusting the focus sliders. Export all loop states minus saturation.
# Each stage uses its own live sliders; unopened widgets use DISPLAY_* and zero shifts.
import importlib
focus_library = str(HERE / "library")
if focus_library not in sys.path:
    sys.path.insert(0, focus_library)
focus_fth = importlib.import_module("fth_reconstruction")  # Same module as rec.focusCDI.
FOCUSED_PNG_OUTPUT = PNG_OUTPUT / "focused_differences"
FOCUSED_PNG_OUTPUT.mkdir(parents=True, exist_ok=True)
focus_stages = {"warmup": primary_warmup, "final": primary_fields}
if refined_fields is not None:
    focus_stages["refined"] = (refined_fields[:, 0] if refined_fields.ndim == 4
                               else refined_fields)
focus_export_records = []
for stage, stage_fields in focus_stages.items():
    sliders = focus_export_sliders.get(stage)
    prop_um, phase_rad, dx, dy = (
        tuple(float(slider.value) for slider in sliders) if sliders is not None
        else (float(DISPLAY_PROP_UM), float(DISPLAY_PHASE_RAD), 0.0, 0.0)
    )
    print(f"{stage}: focus={prop_um:g} µm, phase={phase_rad:g}, dx={dx:g}, dy={dy:g}",
          "(live sliders)" if sliders is not None else "(DISPLAY defaults; no widget opened)")
    focused_reference = focus_fth.reconstructCDI(
        focus_fth.propagate(stage_fields[0], prop_um * 1e-6, DISPLAY_SETUP)
        * np.exp(1j * phase_rad)
    )
    for observation in range(1, len(stage_fields)):
        focused_state = focus_fth.reconstructCDI(
            focus_fth.propagate(stage_fields[observation], prop_um * 1e-6, DISPLAY_SETUP)
            * np.exp(1j * phase_rad)
        )
        focused_difference = np.nan_to_num(
            focused_state - focused_reference, nan=0, posinf=0, neginf=0,
        )
        focused_difference_roi = focus_fth.sub_pixel_centering(
            focused_difference, dx, dy,
        )[DIAGNOSTIC_ROI]
        if not focused_difference_roi.size:
            raise ValueError("DIAGNOSTIC_ROI is empty")
        for component_name, component in (("real", np.real), ("imaginary", np.imag)):
            values = component(focused_difference_roi)
            low, high = np.percentile(values, (1, 99))  # focusCDI(scale=(1, 99), mask=1)
            filename = f"{stage}_{state_labels[observation]}_minus_saturated_{component_name}.png"
            plt.imsave(FOCUSED_PNG_OUTPUT / filename, values, cmap="gray",
                       vmin=low, vmax=high, origin="upper")
            focus_export_records.append(dict(
                file=filename, stage=stage, observation=observation,
                focus_um=prop_um, phase_rad=phase_rad, dx=dx, dy=dy,
                settings_source="live sliders" if sliders is not None else "display defaults",
                component=component_name, vmin=float(low), vmax=float(high),
            ))
(FOCUSED_PNG_OUTPUT / "display_settings.json").write_text(json.dumps({
    "operation": "focused state minus focused saturation; shift then ROI",
    "scale_percentiles": [1, 99], "experimental_setup": DISPLAY_SETUP,
    "roi": [[part.start, part.stop, part.step] for part in DIAGNOSTIC_ROI],
    "images": focus_export_records,
}, indent=2))
print(f"Saved {len(focus_export_records)} focused real/imaginary PNGs to {FOCUSED_PNG_OUTPUT}")


## Inspect relative magnetic and charge-like components

In [ ]:
plt.close("all")

fig,ax=plt.subplots(2,magnetization.shape[0]//2)
for i in range(magnetization.shape[0]):
    ax[i%2,i//2].imshow(np.fft.ifftshift(magnetization[i])[DIAGNOSTIC_ROI], cmap="gray")
    ax[i%2,i//2].axis("off")

In [ ]:
holograms.shape

In [ ]:
DIAGNOSTIC_ROI

In [ ]:
plt.close("all")
common = np.asarray(components["common_log_objects"])[0]
magnetization = np.asarray(components["magnetization"])
magnetic_response = np.asarray(components["magnetic_response"])[0]
relative = (magnetization[1:] - magnetization[0]) * magnetic_response
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(np.abs(np.exp(np.fft.ifftshift(common)))[DIAGNOSTIC_ROI], cmap="gray")
axes[0].set_title("Common charge-like exit wave")
axes[1].imshow(np.fft.ifftshift(magnetization[1])[DIAGNOSTIC_ROI], cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title(f"Reduced magnetization: point {loop_point_indices[indices[0]]}")
axes[2].imshow(np.abs(np.fft.ifftshift(relative[0]))[DIAGNOSTIC_ROI], cmap="magma")
axes[2].set_title("Magnetic change from saturation")
for ax in axes: ax.axis("off")
plt.tight_layout()
fig, ax = plt.subplots(figsize=(8, 4))
# The returned magnetization maps use the shifted object convention.
object_aperture = universal.largest_support_component(components["supportmask_used"])
object_pixels = np.fft.fftshift(object_aperture) != 0
mean_magnetization = np.nanmean(magnetization[1:, object_pixels], axis=1)
ax.plot(loop_current[indices], mean_magnetization, "o-")
ax.set(xlabel="Magnet current (A)", ylabel="Mean reduced magnetization in object aperture")
plt.show()

## Save the retrieval and its state labels

### Cobalt thickness and magnetization normalization (postprocessing)
Set `COBALT_THICKNESS_NM` to the actual total cobalt thickness; no thickness is guessed. The material map used in the fit must be dimensionless relative thickness. With `PHASE_PROPAGATION_SIGN=-1`, use `n=1-delta-i*beta` and transmission `exp(-i*k*(n-1)*d)`. The fitted coefficient is `q=-k*d*beta+i*k*d*delta`, so `beta=-Re(q)/(k*d)` and `delta=Im(q)/(k*d)`. Here `k=2*pi/lambda`; the fitted q already contains k times the reference thickness. Set +1 only for the opposite propagation convention (which reverses the delta conversion). These are effective Co coefficients only if the fitted response is attributable to Co; multilayer contributions and illumination/charge ambiguity need separate calibration.

`NORMALIZE_MAGNETIZATION_TO_UNIT_RANGE=True` divides all state maps by a single global maximum absolute magnetization inside material and multiplies the magnetic spectrum by that same factor. The magnetic product and exit waves are unchanged. This sets max |m| to 1, not both endpoints independently; it is a saturation assumption/convention, not evidence of saturation. Leave False if the fitted scale is already anchored or the observed states may be unsaturated. The solver itself still bounds fitted m to [-1,1]; this option only changes final reporting. Original components remain intact, and converted results are saved separately.


In [ ]:
from library.retrieval_optical_constants import calibrate_responses
# The fitted thickness map is relative: physical thickness = Co thickness * relative map.
optical_calibration = calibrate_responses(
    components["charge_response"], components["magnetic_response"],
    components["magnetization"], np.asarray(components["energy_names"], dtype=float),
    cobalt_thickness_nm=COBALT_THICKNESS_NM,
    propagation_sign=PHASE_PROPAGATION_SIGN,
    normalize_magnetization=NORMALIZE_MAGNETIZATION_TO_UNIT_RANGE,
    material_mask=np.asarray(components["material_thickness"]) > 0,
)
print("Magnetization scale transferred to magnetic spectrum:", optical_calibration["magnetization_scale"])
print("This is a post-fit convention; absolute optical constants also require an anchored illumination/charge gauge.")
calibrated_magnetization = optical_calibration["magnetization"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
energy = optical_calibration["energies_ev"]
order = np.argsort(energy)
for ax, kind in zip(axes, ("charge", "magnetic")):
    if COBALT_THICKNESS_NM is None:
        q = optical_calibration[kind + "_response"]
        ax.plot(energy[order], q.real[order], "o-", label="Re(log response)")
        ax.plot(energy[order], q.imag[order], "o-", label="Im(log response)")
        ax.set_ylabel("Dimensionless log response")
    else:
        ax.plot(energy[order], optical_calibration[kind + "_delta"][order], "o-", label="delta")
        ax.plot(energy[order], optical_calibration[kind + "_beta"][order], "o-", label="beta")
        ax.set_ylabel("Refractive-index coefficient (assumed calibration)")
    ax.set_title(kind.capitalize()); ax.set_xlabel("Energy (eV)"); ax.legend(); ax.grid()
fig.tight_layout()
fig, ax = plt.subplots()
ax.imshow(np.fft.ifftshift(calibrated_magnetization[0]), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Postprocessed magnetization: first state")
plt.show()


In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT, "w") as h:
    if support_transfer_source is not None:
        h.attrs["support_from_01"] = support_transfer_source
        h.attrs["support_adjustment_json"] = json.dumps(support_adjustment)
        h.create_dataset("support_input_from_01", data=support_input_from_01, compression="gzip")
    h.attrs["preprocess_config_json"] = json.dumps(cache_config)
    h.attrs["source_cache"] = str(CACHE)
    calibration_group = h.create_group("optical_calibration")
    for key, value in optical_calibration.items():
        if np.ndim(value) == 0:
            calibration_group.attrs[key] = value
        else:
            calibration_group.create_dataset(key, data=value)
    if refined_fields is not None:
        h.create_dataset("poisson_refined_fields", data=refined_fields, compression="gzip")
        refinement_group = h.create_group("poisson_refinement")
        refinement_group.attrs["description"] = "Quasi-Poisson camera-unit polish; fixed gamma and secondary modes. Physical components describe baseline fields only."
        refinement_group.attrs["steps"] = POISSON_STEPS
        refinement_group.attrs["learning_rate"] = POISSON_LEARNING_RATE
        for obs, record in enumerate(poisson_records):
            group = refinement_group.create_group(str(obs))
            group.attrs["status"] = record["status"]
            for key in ("baseline_loss", "valid_pixels", "zero_pixels"):
                if key in record:
                    group.attrs[key] = record[key]
            group.create_dataset("loss", data=record["loss"])
            group.create_dataset("step_sizes", data=record["step_sizes"])
        refinement_group.attrs["stage_order"] = "warmup,final,refined; all evaluated with final gamma"
        refinement_group.create_dataset("stage_deviance", data=poisson_stage_deviance)
    h.attrs["support_source_shift"] = support_source_shift
    h.attrs["mode_support_shifts"] = components["mode_support_shifts"]
    h.attrs["workflow"] = "aperiodic_hysteresis_universal_physical"
    h.attrs["detector_constraint_diagnostics_json"] = json.dumps(errors.get("detector_constraint_diagnostics", []))
    h.attrs["execution_summary_json"] = json.dumps(errors.get("execution_summary", {}))
    h.attrs["warmup_pass_timings_json"] = json.dumps(errors.get("warmup_passes", []))
    recipe_metadata = dict(recipe)
    for key in ("material_mask", "material_thickness"):
        if recipe_metadata[key] is not None:
            array = np.asarray(recipe_metadata[key])
            h.create_dataset("input_" + key, data=array, compression="gzip")
            recipe_metadata[key] = {"hdf5_dataset": "input_" + key,
                                    "shape": list(array.shape)}
    h.attrs["recipe_json"] = json.dumps(
        recipe_metadata, default=lambda x: np.asarray(x).tolist())
    h.attrs["photon_energy_eV"] = ENERGY_EV
    h.attrs["saturated_sign"] = SATURATED_SIGN
    h.attrs["hologram_offset"] = float(HOLOGRAM_OFFSET)
    for name, value in {"fields":fields, "warmup_fields":warmup,
                        "supportmask":components["supportmask"],
                        "modal_supportmask_used":components["modal_supportmask_used"],
                        "mask_pixel":components["mask_pixel"], "bsmasks":bsmasks,
                        "common_log_objects":components["common_log_objects"],
                        "magnetization":magnetization,
                        "material_thickness":components["material_thickness"],
                        "magnetic_response":components["magnetic_response"],
                        "charge_response":components["charge_response"],
                        "loop_indices":loop_point_indices[indices],
                        "loop_cache_positions":np.asarray(indices),
                        "magnet_current_A":loop_current[indices],
                        "intensity_factors":intensity_factors,
                        "fitted_intensity_offsets":fitted_offsets,
                        **({"coherence_kernel": components["coherence_kernel"]}
                           if components["partial_coherence"]
                           and COHERENCE_KERNEL_SCOPE == "shared" else {}),
                        **({"coherence_kernels": components["coherence_kernels"]}
                           if components["partial_coherence"]
                           and COHERENCE_KERNEL_SCOPE == "per_observation" else {})}.items():
        h.create_dataset(name, data=np.asarray(value), compression="gzip" if np.ndim(value)>1 else None)
    h.create_dataset("state_labels", data=np.asarray(state_labels, dtype="S"))
print("Saved", OUTPUT)

In [ ]:
plt.close("all")

In [ ]:
fig,ax=plt.subplots()
ax.imshow(holograms[0]*(mask_pixel[0]))